In [ ]:
import os
import json
import glob
import matplotlib.pyplot as plt

LOGDIR = "outputs/cifar_fcn"
optimizer = "sgd"
activation = "relu"
inits = ["gaussian", "identity"]
depths = [2, 3, 5]
seed = 0

# depth별 색 고정
cmap = plt.get_cmap("tab10")
depth_to_color = {d: cmap(i) for i, d in enumerate(depths)}

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
ax_layer = axes[1]
ax_loss = axes[0]

for depth in depths:
    color = depth_to_color[depth]

    for init_type in inits:
        linestyle = "-" if init_type == "gaussian" else "--"
        marker = "o" if init_type == "gaussian" else "x"

        pattern = os.path.join(
            LOGDIR,
            f"{optimizer}_{init_type}_{activation}_depth{depth}_seed{seed}.json",
        )
        matches = glob.glob(pattern)
        if not matches:
            print(f"File not found for depth={depth}, init={init_type}: pattern={pattern}")
            continue

        path = matches[0]
        with open(path, "r") as f:
            data = json.load(f)

        epoch_stats = data["epoch_stats"]

        epochs = [e["epoch"] for e in epoch_stats]

        # 마지막 레이어 제외 평균 stable rank
        avg_layer_stable_ranks = []
        for e in epoch_stats:
            layers = e["layers"]
            if len(layers) <= 1:
                layer_subset = layers
            else:
                layer_subset = layers[:-1]

            vals = [l["stable_rank"] for l in layer_subset]
            avg_layer_stable_ranks.append(sum(vals) / len(vals))

        # train loss
        train_losses = [e["train_loss"] for e in epoch_stats]

        # 여기서 5의 배수 epoch만 선택 (5, 10, 15, ...)
        plot_indices = [i for i, ep in enumerate(epochs) if ep % 5 == 0]
        if not plot_indices:
            continue

        epochs_sub = [epochs[i] for i in plot_indices]
        avg_layer_stable_ranks_sub = [avg_layer_stable_ranks[i] for i in plot_indices]
        train_losses_sub = [train_losses[i] for i in plot_indices]

        label_suffix = "Gaussian" if init_type == "gaussian" else "Identity"
        label = rf"$L={depth}$, {label_suffix}"

        ax_layer.plot(
            epochs_sub,
            avg_layer_stable_ranks_sub,
            linestyle=linestyle,
            color=color,
            marker=marker,
            label=label,
        )
        ax_loss.plot(
            epochs_sub,
            train_losses_sub,
            linestyle=linestyle,
            color=color,
            marker=marker,
            label=label,
        )

ax_layer.set_xlabel("Epoch")
ax_layer.set_ylabel("AVG Stable Rank")
ax_layer.grid(True)
ax_loss.set_yscale('log')
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Train Loss")
ax_loss.grid(True)
ax_loss.legend(fontsize=10.)

plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/gaussian_identity.pdf')
plt.show()
